# Day 1.1 — Your First Model Call

We begin with the smallest useful AI application:

```text
Question  ->  Model  ->  Text response
```

This is **not yet an agent**. Nothing loops, nothing runs on your machine, and nothing
decides what to do next. Today we add each of those layers one at a time — but first we
have to be able to send a single request and read what comes back.

## Before you begin

### Learning outcomes

- Create the `.env` file that every notebook in this course reads, and verify it without
  ever printing your key in full.
- Send one prompt through the classroom route and name each field of the request.
- Read the usage record and explain what a *context window* is.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

With no key you stay in MOCK mode and every cell below still prints a result. With a key,
the wording of the answer changes from run to run, but the request fields and the usage
record stay in exactly the same shape.

## Concept briefing

## Why this day exists

A language model is a generator, not an application. It receives a finite context and
predicts a continuation. It does not automatically know your files, execute Python, or
continue working until a goal is complete. An agentic application is created when host
code gives the model a limited set of possible actions, carries state between turns,
executes approved actions, and decides when the run must stop.

Day 1 removes the apparent magic from this process. By the end, students should be able
to point to the exact line that sends a request, the exact data that describes a tool,
the exact function that executes it, and the exact condition that terminates the loop.

## What a model call actually contains

A typical request contains a model identifier, ordered messages, optional tool schemas,
and generation controls. Messages are not merely a chat transcript. Their roles tell the
provider how each piece should be interpreted:

- `system`: standing instructions and boundaries;
- `user`: the current task or supplied information;
- `assistant`: previous model output, including tool requests;
- `tool`: an observation produced by host-executed code.

Two request fields deserve names of their own. `temperature` controls how freely the
model samples its next token: 0 makes it take the most likely continuation every time,
which is what pipelines and tool selection need; higher values trade reproducibility for
variety. `max_tokens` bounds only the generated answer, and on reasoning models it is
spent on private reasoning tokens before any visible output, which is why a small limit
can truncate a JSON response mid-field.

Everything the model can see in one call - system message, every earlier message, tool
descriptions and the answer being generated - must fit inside its **context window**, a
fixed token budget. Nothing carries over between calls: a model is stateless, and the
appearance of memory comes from the application re-sending the conversation each time.
This is also why a growing conversation costs more per turn, and why Day 3 has to manage
history rather than let it accumulate.

The provider serializes this request into a form the model can process. The model sees
tokens representing instructions, messages and tool descriptions. It does not receive a
live Python function. When it appears to "call" a tool, it is generating structured
tokens that name a function and propose arguments. The host application parses those
tokens, validates the arguments, applies policy, calls ordinary code, and returns the
result in another message.

This distinction is load-bearing:

```text
model proposes structured tokens
-> application validates and authorizes
-> Python executes
-> application records the observation
-> model sees the observation on the next call
```

If the model invents a tool name, supplies the wrong type, or requests a prohibited
action, nothing should happen unless the application accepts the request.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Setting up your .env file

Your API key is a password. It must never appear inside a notebook, a screenshot, or a
commit. The course keeps it in one file called `.env` that Python reads at run time.

**Where does it go?** In the **repository root** — the folder that already contains
`README.md` and `.env.example`, not inside `day_01_model_tools_agent/`.

**Windows (PowerShell), from the repository root:**

```powershell
Copy-Item .env.example .env
notepad .env
```

**macOS / Linux, from the repository root:**

```bash
cp .env.example .env
nano .env      # or: open -e .env
```

**The name matters.** The file is called exactly `.env` — a dot, then `env`, and *no*
extension. Windows Explorer likes to save it as `.env.txt`; if the setup cell cannot find
your key, that is almost always why. Turn on *File name extensions* in Explorer and check.

**What goes inside.** Fill in these two lines (leave the rest of the copied file alone):

```dotenv
OPENROUTER_API_KEY=sk-or-...your own issued key...
OPENROUTER_MODEL=openai/gpt-oss-120b
```

**Never commit it.** `.env` is already listed in `.gitignore`. `.env.example` is the
template that *is* committed and contains no secret. Your key has a course-wide lifetime
spending limit — do not share it and do not paste it into a notebook cell.

**On Google Colab** there is no repository folder to edit, so the day notebook starts with
a *Colab bootstrap* cell instead. It asks for the key with `getpass` (which hides your
typing) and stores it in `os.environ` for that runtime only — nothing is written to disk,
and the key disappears when the runtime is recycled:

```python
import os, getpass
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter key: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-oss-120b"
```

If you have no key at all, do nothing: every notebook runs in MOCK mode.

In [ ]:
# --- Verify the .env file without leaking the key ------------------------------
# os.getenv returns None when a variable is missing. We never use os.environ["..."]
# because that raises KeyError and would stop the whole notebook.
api_key = os.getenv("OPENROUTER_API_KEY")
model_id = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

def mask(secret):
    """Print just enough of a secret to recognise it, never the whole thing."""
    if not secret:
        return "(not set)"
    if len(secret) < 12:
        return "(set, but suspiciously short - did you paste the whole key?)"
    return f"{secret[:6]}...{secret[-4:]}"      # e.g. sk-or-...9f2a

print("OPENROUTER_API_KEY :", mask(api_key))
print("OPENROUTER_MODEL   :", model_id)
print("Mode               :", "LIVE" if api_key else "MOCK (every cell below still runs)")

### Step 1 — Build the client, but only if a key exists

Constructing a client (or a provider object) unconditionally is the classic way to make a
notebook crash on its first cell for every student without a key. We build it inside an
`if`, and print which route we ended up on.

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 2 — What is actually in a model call?

A "model call" is an HTTP request carrying a small dictionary. Building that dictionary
*before* sending it makes the moving parts visible: which model, which messages, how long
the answer may be, and how random it may be.

In [ ]:
# The request is ordinary Python data. Nothing is sent yet.
question = "Explain recursion in two sentences for a beginner."

request_payload = {
    "model": model_id,                  # which model should answer
    "messages": [                       # the whole conversation, oldest first
        {"role": "user", "content": question},
    ],
    "max_tokens": 300,                  # upper bound on the ANSWER length
    "temperature": 0,                   # 0 = pick the most likely next word every time
}

print("Fields the provider will receive:")
for field, value in request_payload.items():
    print(f"  {field:12} = {value}")

### Step 3 — Send the request and read the answer

`send_chat` below is the only place in this notebook that touches the network, and it does
so only when a key was found. Otherwise it returns a fixed mock answer, so the lesson still
runs. Any live error is caught and reported in one line instead of ending the class.

In [ ]:
MOCK_ANSWER = (
    "Recursion is when a function solves a problem by calling itself on a smaller "
    "version of the same problem. It needs a base case, otherwise the calls never stop."
)

def send_chat(messages, max_tokens=300, temperature=0):
    """Return (answer_text, usage_dict). Falls back to the mock on any problem."""
    if client is None:
        # MOCK route: deterministic text, zero tokens, zero cost.
        return MOCK_ANSWER, {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            # "reasoning" is an OpenRouter extra: keep the model's private thinking short
            # and out of the reply, because we only asked for two sentences.
            extra_body={"reasoning": {"effort": "low", "exclude": True}},
        )
        usage = response.usage
        return response.choices[0].message.content, {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "cost_usd": getattr(usage, "cost", 0.0) or 0.0,
        }
    except Exception as exc:
        print("Live call failed, using the mock answer instead ->", type(exc).__name__, exc)
        return MOCK_ANSWER, {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}

answer, usage = send_chat(request_payload["messages"])
print("Question :", question)
print("Answer   :", answer)

### Step 4 — Read the usage record

Every live response reports how many tokens went in and came out. Tokens are the billing
unit: roughly 4 characters of English each. In MOCK mode all counters are zero, which is
itself useful — you can tell at a glance that no credit was spent.

In [ ]:
print("Usage reported by the provider:")
for field, value in usage.items():
    print(f"  {field:18} = {value}")

# A rule of thumb you can apply without any API: about 4 characters per token.
estimated_prompt_tokens = len(question) // 4
print()
print("Rough estimate of the prompt size :", estimated_prompt_tokens, "tokens")
print("Answer length in characters       :", len(answer))
print("Did the model run any Python here?: no - it only generated text")

### Step 5 — The context window

The **context window** is the maximum number of tokens a model can look at in one call:
system message + every earlier message + tool descriptions + the answer being generated.
It is a hard limit, not a suggestion.

Two consequences you will meet all week:

- Nothing is remembered between calls. Each request must carry everything the model needs.
- A conversation that grows forever eventually stops fitting, so Day 3 has to *manage* it.

In [ ]:
# Pretend we keep appending turns to one conversation and never trim it.
CONTEXT_WINDOW_TOKENS = 128_000   # typical for the classroom model; check your model card
tokens_per_turn = 220             # a small question plus a short answer

conversation_tokens = 0
for turn in range(1, 6):
    conversation_tokens += tokens_per_turn
    percent = 100 * conversation_tokens / CONTEXT_WINDOW_TOKENS
    print(f"after turn {turn}: {conversation_tokens:>5} tokens  ({percent:.2f}% of the window)")

turns_until_full = CONTEXT_WINDOW_TOKENS // tokens_per_turn
print()
print("Turns before this conversation stops fitting:", turns_until_full)
print("Cost per call also grows, because the WHOLE history is re-sent every time.")

### Step 6 — The other routes (optional, read-only)

The guided notebooks all use OpenRouter. Two alternatives exist and are shown here for
completeness; you need neither to finish Day 1.

- **Ollama** runs a small model on your own machine. Useful later for comparing model
  capability and latency; it needs a reasonably powerful computer.
- **Direct OpenAI API** is for students who already have their own OpenAI project. A
  ChatGPT subscription and API billing are separate things.

Both cells below are deliberately commented out.

In [ ]:
# --- Optional route A: a local model through Ollama ----------------------------
# %pip install -q ollama
# from ollama import chat
# local = chat(model="qwen3:4b", messages=[{"role": "user", "content": question}])
# print(local.message.content)

# --- Optional route B: your own OpenAI account ---------------------------------
# The SDK reads OPENAI_API_KEY from the environment. Put the model id you actually
# have access to in .env, for example:
#     OPENAI_MODEL=<model id from your OpenAI account>
# from openai import OpenAI
# direct_client = OpenAI()                       # reads OPENAI_API_KEY
# direct = direct_client.responses.create(
#     model=os.getenv("OPENAI_MODEL", "<model id from your OpenAI account>"),
#     input=question,
# )
# print(direct.output_text)

print("Both optional routes are commented out on purpose - nothing ran.")

### Try it yourself

`max_tokens` bounds the **answer**, not the question. Predict what happens if we ask the
same question with `max_tokens=12`, then run the cell below and compare.

In [ ]:
# --- Worked solution ---
# Prediction: in LIVE mode the answer is cut off mid-sentence, because max_tokens is a
# hard stop applied while the model is generating - it is NOT a polite request to be
# brief. In MOCK mode nothing is generated, so our fixed string comes back untouched;
# that difference is a good reminder of what a mock can and cannot teach you.

short_answer, short_usage = send_chat(
    [{"role": "user", "content": question}],
    max_tokens=12,
)
print("max_tokens=12 ->", short_answer)
print("completion_tokens:", short_usage["completion_tokens"])
print()
if client is None:
    print("MOCK mode: the limit was accepted but never applied, because no model ran.")
else:
    print("LIVE mode: notice the sentence simply stops. The model was interrupted.")

## Required live observation

Send one bounded prompt through the issued OpenRouter route and save the response plus usage record. If service access fails, inspect the instructor-captured trace and continue in mock mode.


### Checkpoint

**1. Your setup cell prints `MOCK (no OPENROUTER_API_KEY found)` but you are sure you created the file. What are the two most likely causes?**

<details><summary>Show answer</summary>

1. The file is not really called `.env` — Windows saved it as `.env.txt`. Turn on file
   name extensions and rename it.
2. It is in the wrong folder. It must sit in the **repository root**, beside `README.md`
   and `.env.example`, not inside `day_01_model_tools_agent/`.

A third, rarer cause: the line was written as `OPENROUTER_API_KEY = sk-or-...` with spaces
around the `=`, or the key was wrapped in quotes it does not need.

</details>

**2. The model answered a question about recursion. Did it run any Python to do that, and where did the answer's length limit come from?**

<details><summary>Show answer</summary>

No Python ran. The model only produced text, one token at a time — that is all a model
call does. The length limit came from `max_tokens` in the request *we* built, not from
anything the model chose; and the total of everything sent plus everything generated must
fit inside the model's context window.

</details>

### Recap

- **Limitation we saw:** a model call returns free-form text, remembers nothing between
  calls, and cannot act on your machine.
- **Layer we added:** a reproducible request — `.env` for the key, an explicit payload of
  model, messages, `max_tokens` and `temperature`, and a mock fallback so the lesson never
  depends on the network.
- **Evidence it worked:** the masked-key check printed the mode, the answer printed, and
  the usage record showed exactly what was billed (zero in mock mode).